# Exercise 1 · Batch Pipeline — Customer RFM Segmentation

## The problem

You have 6 months of e-commerce transactions. Your goal is to build a batch pipeline that segments customers using the **RFM model**:

| Metric | Question | How to compute |
|---|---|---|
| **R**ecency | How recently did they buy? | Days since last purchase |
| **F**requency | How often do they buy? | Number of orders |
| **M**onetary | How much do they spend? | Total spend |

Each metric is scored 1–5 (5 = best). Scores are combined to classify customers into segments like *Champions*, *Loyal*, *At Risk*, and *Lost*.

## Pipeline
```
Generate data  →  Compute RFM  →  Score 1-5  →  Segment
      ↓                                              ↓
  PostgreSQL                               Parquet + DuckDB
```

## Skills practiced
- Multi-source reads (JDBC + in-memory)
- `groupBy` + `agg` for metric computation
- `ntile()` window function for percentile scoring
- JDBC write to PostgreSQL
- DuckDB for final analytics and visualisation

---
## Step 0 · Setup

In [ ]:
import os
from datetime import date, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
import random
from faker import Faker

SPARK_MASTER = os.getenv('SPARK_MASTER', 'spark://spark-master:7077')
PG_URL   = f"jdbc:postgresql://{os.getenv('POSTGRES_HOST','postgres')}:5432/{os.getenv('POSTGRES_DB','sandbox')}"
PG_PROPS = {'user': os.getenv('POSTGRES_USER','spark'),
            'password': os.getenv('POSTGRES_PASSWORD','spark123'),
            'driver': 'org.postgresql.Driver'}

spark = (
    SparkSession.builder
    .appName('rfm-pipeline')
    .master(SPARK_MASTER)
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version}  master: {spark.sparkContext.master}')

---
## Step 1 · Generate synthetic transactions

We generate 500 customers with 6 months of purchases.
Some are frequent buyers, some churned long ago — this makes the segmentation interesting.

In [ ]:
fake = Faker()
Faker.seed(0)
random.seed(0)

TODAY      = date(2026, 6, 17)
START_DATE = TODAY - timedelta(days=180)
N_CUSTOMERS = 500
CATEGORIES  = ['electronics', 'clothing', 'food', 'travel', 'health']

# Each customer has a "persona" that shapes their buying pattern
def customer_persona(cid):
    if cid % 5 == 0:  return 'champion'    # frequent, recent, high spend
    if cid % 5 == 1:  return 'loyal'       # frequent, recent, medium spend
    if cid % 5 == 2:  return 'at_risk'     # used to buy, went quiet
    if cid % 5 == 3:  return 'new'         # one recent purchase
    return 'lost'                           # bought once, long ago

rows = []
for cid in range(1, N_CUSTOMERS + 1):
    persona = customer_persona(cid)
    if persona == 'champion':
        n_orders, recency_cap, spend_range = 20, 14,  (100, 800)
    elif persona == 'loyal':
        n_orders, recency_cap, spend_range = 10, 30,  (40,  300)
    elif persona == 'at_risk':
        n_orders, recency_cap, spend_range =  5, 180, (20,  200)
    elif persona == 'new':
        n_orders, recency_cap, spend_range =  1, 7,   (30,  150)
    else:  # lost
        n_orders, recency_cap, spend_range =  1, 180, (10,  100)

    last_purchase_days_ago = random.randint(1, recency_cap)
    for _ in range(n_orders):
        order_date = TODAY - timedelta(
            days=random.randint(last_purchase_days_ago, 180)
        )
        if order_date < START_DATE:
            continue
        rows.append((
            cid,
            fake.name(),
            fake.country_code(),
            round(random.uniform(*spend_range), 2),
            random.choice(CATEGORIES),
            order_date.isoformat(),
        ))

schema = StructType([
    StructField('customer_id',  IntegerType()),
    StructField('customer_name',StringType()),
    StructField('country',      StringType()),
    StructField('amount',       DoubleType()),
    StructField('category',     StringType()),
    StructField('order_date',   StringType()),
])

raw = spark.createDataFrame(rows, schema).withColumn('order_date', F.to_date('order_date'))
print(f'Total orders: {raw.count():,}   Distinct customers: {raw.select("customer_id").distinct().count()}')
raw.show(5)

---
## Step 2 · Compute RFM metrics

Per customer we need:
- **R** = days between `TODAY` and the most recent `order_date`
- **F** = count of distinct orders
- **M** = sum of `amount`

In [ ]:
snapshot = F.to_date(F.lit(TODAY.isoformat()))

rfm = (
    raw.groupBy('customer_id', 'customer_name', 'country')
    .agg(
        F.datediff(snapshot, F.max('order_date')).alias('recency'),    # lower = better
        F.count('*').alias('frequency'),                                # higher = better
        F.round(F.sum('amount'), 2).alias('monetary'),                  # higher = better
    )
)

rfm.orderBy('recency').show(10)

---
## Step 3 · Score each metric 1–5 using `ntile()`

`ntile(5)` splits the population into 5 equal buckets ordered by the metric.
**Important:** recency is reversed — a *lower* number of days is *better*, so we order descending.

In [ ]:
# Window specs — ntile needs an ORDER BY
w_r = Window.orderBy(F.col('recency').asc())     # fewer days → higher score (5)
w_f = Window.orderBy(F.col('frequency').asc())
w_m = Window.orderBy(F.col('monetary').asc())

scored = (
    rfm
    .withColumn('r_score', F.ntile(5).over(w_r))
    .withColumn('f_score', F.ntile(5).over(w_f))
    .withColumn('m_score', F.ntile(5).over(w_m))
    # Invert recency: ntile(1) = smallest recency (best) → should be score 5
    .withColumn('r_score', F.lit(6) - F.col('r_score'))
    .withColumn('rfm_score', F.col('r_score') + F.col('f_score') + F.col('m_score'))
)

scored.select('customer_id', 'recency', 'frequency', 'monetary',
              'r_score', 'f_score', 'm_score', 'rfm_score') \
      .orderBy(F.col('rfm_score').desc()) \
      .show(10)

---
## Step 4 · Assign segments

Map the combined RFM score (3–15) to human-readable segments.

| Score range | Segment |
|---|---|
| 13–15 | Champions |
| 10–12 | Loyal |
| 7–9 | Needs Attention |
| 4–6 | At Risk |
| 3 | Lost |

In [ ]:
segmented = scored.withColumn(
    'segment',
    F.when(F.col('rfm_score') >= 13, 'Champions')
     .when(F.col('rfm_score') >= 10, 'Loyal')
     .when(F.col('rfm_score') >= 7,  'Needs Attention')
     .when(F.col('rfm_score') >= 4,  'At Risk')
     .otherwise('Lost')
)

# Distribution check
segmented.groupBy('segment') \
         .agg(F.count('*').alias('customers'),
              F.round(F.avg('monetary'), 2).alias('avg_spend')) \
         .orderBy('customers', ascending=False) \
         .show()

---
## Step 5 · Write results

In [ ]:
# Write to PostgreSQL
(
    segmented
    .select('customer_id', 'customer_name', 'country',
            'recency', 'frequency', 'monetary',
            'r_score', 'f_score', 'm_score', 'rfm_score', 'segment')
    .write.jdbc(
        PG_URL, 'customer_rfm', mode='overwrite',
        properties={**PG_PROPS,
                    'truncate': 'false',          # full drop+create — schema changes each run
                    'createTableColumnTypes':
                        'customer_id INT, customer_name VARCHAR(120), country CHAR(2), '
                        'recency INT, frequency INT, monetary FLOAT, '
                        'r_score INT, f_score INT, m_score INT, rfm_score INT, '
                        'segment VARCHAR(20)'}
    )
)
print('PostgreSQL ✓')

In [ ]:
# Write to Parquet for DuckDB analysis
segmented.write.mode('overwrite').parquet('/home/jupyter/data/output/rfm')
print('Parquet ✓')

---
## Step 6 · Analyze with DuckDB

In [ ]:
import duckdb, matplotlib.pyplot as plt

con = duckdb.connect()
con.execute("CREATE VIEW rfm AS SELECT * FROM read_parquet('/home/jupyter/data/output/rfm/**/*.parquet')")

# Segment summary
summary = con.execute("""
    SELECT segment,
           COUNT(*)                         AS customers,
           ROUND(AVG(recency), 1)           AS avg_days_since_purchase,
           ROUND(AVG(frequency), 1)         AS avg_orders,
           ROUND(AVG(monetary), 2)          AS avg_spend,
           ROUND(SUM(monetary), 2)          AS total_revenue
    FROM rfm
    GROUP BY ALL
    ORDER BY avg_spend DESC
""").df()

summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

summary.set_index('segment')['customers'].plot(
    kind='bar', ax=axes[0], color='steelblue', title='Customers per Segment')
axes[0].set_ylabel('count')
axes[0].tick_params(axis='x', rotation=30)

summary.set_index('segment')['total_revenue'].plot(
    kind='bar', ax=axes[1], color='darkorange', title='Total Revenue per Segment')
axes[1].set_ylabel('USD')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Frequency vs Monetary coloured by Segment — spot outliers
detail = con.execute("SELECT frequency, monetary, segment FROM rfm").df()

colours = {'Champions':'gold', 'Loyal':'steelblue', 'Needs Attention':'orange',
           'At Risk':'tomato', 'Lost':'grey'}

fig, ax = plt.subplots(figsize=(9, 5))
for seg, grp in detail.groupby('segment'):
    ax.scatter(grp['frequency'], grp['monetary'], label=seg,
               color=colours.get(seg,'black'), alpha=0.5, s=30)

ax.set_xlabel('Frequency (orders)')
ax.set_ylabel('Monetary (total spend)')
ax.set_title('Frequency vs Monetary by Segment')
ax.legend()
plt.tight_layout()
plt.show()

---
## Challenge — try it yourself

1. **Add a category dimension** — which segment buys most electronics vs food? Pivot the result.
2. **Country filter** — re-run the RFM only for `country = 'US'`. Do the segment proportions change?
3. **Stricter Champions** — change the Champions threshold to require `r_score = 5 AND f_score >= 4`. How many remain?
4. **Delta Lake** — replace the Parquet write with a Delta Lake write (`format('delta')`). Then do an UPDATE to set `segment = 'VIP'` for customers with `rfm_score = 15`.